# 05 · Explainability and fairness audit

Reads the outputs of `python -m ssn explain`, `fairness audit`, `fairness mitigate`, and `model-card`. Explanations come from the saved final pipeline on the held-out cohort features; fairness metrics use the evaluator-only labels and verified sensitive-attribute encodings. Sensitive attributes are never model inputs or adviser-facing reasons.

Written interpretation: `reports/bias_fairness_analysis.md`, `reports/limitations.md`, `reports/model_card.md`.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from ssn.paths import repo_root

ROOT = repo_root()
E, FA = ROOT / 'reports' / 'explainability', ROOT / 'reports' / 'fairness'
method = json.loads((E / 'method.json').read_text())
method

## 1. Global importance (SHAP, aggregated per source feature, averaged over the calibrated ensemble members)

In [ ]:
display(Image(str(E / 'shap_global_bar.png')))
pd.read_csv(E / 'shap_global_importance.csv').head(20)

In [ ]:
display(Image(str(E / 'shap_beeswarm.png')))

## 2. Representative local cases (synthetic record ids; supportive language)

One true positive, false positive, false negative, and true negative at the deployed threshold. `adviser_phrases` is what an adviser would see: no sensitive attribute appears.

In [ ]:
examples = json.loads((E / 'shap_local_examples.json').read_text())
for ex in examples:
    if not ex.get('available'):
        print(ex['case'], 'no example available'); continue
    print(f"\n=== {ex['case']} | record {ex['record_id']} | score {ex['score']:.3f} | band {ex['band']} | outcome is_dropout={ex['outcome_is_dropout']} ===")
    for ph in ex['adviser_phrases']:
        print(f"  - {ph['label']}: {ph['phrase']} ({ph['direction']}, strength {ph['strength']:.3f})")

## 3. Partial dependence and ICE (suitable raw continuous features)

In [ ]:
sel = pd.read_csv(E / 'pdp_ice_selection.csv')
display(sel)
for p in sorted(E.glob('pdp_ice_*.png')):
    display(Image(str(p)))

## 4. Fairness audit (aggregate; held-out cohort)

In [ ]:
g = json.loads((FA / 'group_metrics.json').read_text())
print('n =', g['n'], '| threshold =', round(g['threshold'], 4), '| k =', g['k'], '| min group size =', g['min_group_size'])
print('gender encoding', g['gender_encoding'], 'verified:', g['gender_encoding_verified'])
display(pd.DataFrame(g['attribute_summary']).round(4))
pd.DataFrame(g['groups'])[['attribute','operating_point','group','n','reliable','base_rate','selection_rate','tpr','fpr','brier']].round(4)

In [ ]:
display(Image(str(FA / 'selection_rates.png')))
display(Image(str(FA / 'group_calibration.png')))

## 5. Mitigation experiment (training OOF only; the test set is not re-used)

In [ ]:
pd.read_csv(FA / 'mitigation_comparison.csv').drop(columns=['note']).round(4)

## 6. Model card

In [ ]:
display(Markdown((ROOT / 'reports' / 'model_card.md').read_text()))